# 미세먼지 예방·대비를 위한 데이터 분석 연습 (Google Colab · 3-셀 구성)

**목표**: 직접 다운로드한 미세먼지(에어코리아) 자료와 기상(기상청) 자료를 코랩에 업로드해서, "바람이 약하고 비가 안 올 때 미세먼지가 심해지는가"를 실제 수치로 확인하고, 예방·대비용 요약 메시지까지 만들어봅니다.

## 준비물 (직접 다운로드)
1. **미세먼지 자료** — 에어코리아(airkorea.or.kr) "정보공개 → 다운로드" 또는 공공데이터포털에서 원하는 기간·측정소의 PM10(또는 PM2.5) 자료를 CSV로 다운로드
2. **기상 자료** — 기상자료개방포털(data.kma.go.kr) "지상관측자료(ASOS) 시간자료"에서 같은 기간·같은 지역의 풍속·강수량 자료를 CSV로 다운로드

> 두 자료의 **기간과 지역(측정소/지점)을 최대한 맞춰서** 받아야 나중에 시간 기준으로 합칠 수 있습니다.

## 실습 흐름
1. 준비 — 한글 폰트 설치, 라이브러리 임포트
2. 설정 + 데이터 불러오기 — 두 CSV 업로드 → 컬럼 정리 → 하나의 표로 합치기
3. 결과 확인 — 상관관계·그래프·예방 메시지

> ⚠️ **에러가 나면 셀을 하나씩 다시 실행하지 말고, 메뉴에서 "런타임 → 모두 실행(Run all)"을 눌러 처음부터 다시 실행하세요.**
> (중간 셀만 다시 실행하면 이전 실행 결과가 남아있어 오류 원인 파악이 더 어려워집니다.)


## 셀 1. 준비: 한글 폰트 설치 + 라이브러리 임포트

In [ ]:
# 코랩(리눅스) 환경에는 한글 폰트가 기본으로 없어서 나눔고딕을 설치합니다.
!apt-get -qq -y install fonts-nanum > /dev/null

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)
plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False  # 마이너스(-) 기호 깨짐 방지

print("준비 완료. (그래프 글자가 네모(ㅁ)로 깨지면 '런타임 > 런타임 다시 시작' 후 전체를 다시 실행)")

## 셀 2. 설정 + 데이터 불러오기

두 파일을 업로드받고, 컬럼 이름을 정리해서 하나의 표로 합치는 것까지 이 셀 하나로 끝냅니다.

**아래에서 직접 수정해야 하는 곳은 `AIR_COL_MAP`과 `WEATHER_COL_MAP` 두 딕셔너리뿐입니다** — 본인이 받은 파일을 먼저 열어(또는 아래 `head()` 출력을 보고) 실제 컬럼명이 다르면 왼쪽 값만 그 이름으로 바꿔주세요.

In [ ]:
# -------------------- 2-1. 파일 업로드 --------------------
from google.colab import files

print("1) 미세먼지(에어코리아) 파일을 선택하세요.")
uploaded_air = files.upload()
print("\n2) 기상관측(기상청 ASOS) 파일을 선택하세요.")
uploaded_weather = files.upload()

air_filename = list(uploaded_air.keys())[0]
weather_filename = list(uploaded_weather.keys())[0]

# -------------------- 2-2. 파일 읽기 (인코딩 자동 시도) --------------------
# 공공기관 CSV는 한글 인코딩이 cp949(EUC-KR)인 경우가 많아 순서대로 시도합니다.
def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "utf-8"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"{path} 인코딩을 확인해주세요. (엑셀에서 열어 'CSV UTF-8'로 다시 저장해도 됩니다)")

df_air_raw = read_csv_safe(air_filename)
df_weather_raw = read_csv_safe(weather_filename)

print("=== 미세먼지 파일 컬럼 ===")
print(df_air_raw.columns.tolist())
display(df_air_raw.head())

print("\n=== 기상 파일 컬럼 ===")
print(df_weather_raw.columns.tolist())
display(df_weather_raw.head())

# -------------------- 2-3. 컬럼 이름 정리 --------------------
# 에어코리아 다운로드 파일에 흔히 쓰이는 컬럼명 예시: "측정일시", "PM10", "PM25"(또는 "PM2.5")
# ↓ 왼쪽(키)을 실제 파일의 컬럼명으로, 오른쪽(값)은 그대로 두세요.
AIR_COL_MAP = {
    "측정일시": "일시",
    "PM10": "PM10",
}

# 기상자료개방포털(ASOS) 다운로드 파일에 흔히 쓰이는 컬럼명 예시: "일시", "풍속(m/s)", "강수량(mm)"
WEATHER_COL_MAP = {
    "일시": "일시",
    "풍속(m/s)": "풍속",
    "강수량(mm)": "강수량",
}

df_air = df_air_raw.rename(columns=AIR_COL_MAP)[list(AIR_COL_MAP.values())]
df_weather = df_weather_raw.rename(columns=WEATHER_COL_MAP)[list(WEATHER_COL_MAP.values())]

# -------------------- 2-4. 시간 형식 통일 + 병합 --------------------
df_air["일시"] = pd.to_datetime(df_air["일시"], errors="coerce")
df_weather["일시"] = pd.to_datetime(df_weather["일시"], errors="coerce")

# 강수량은 비가 안 온 시간이 빈칸(NaN)으로 들어오는 경우가 많아 0으로 채웁니다.
df_weather["강수량"] = pd.to_numeric(df_weather["강수량"], errors="coerce").fillna(0)
df_air["PM10"] = pd.to_numeric(df_air["PM10"], errors="coerce")
df_weather["풍속"] = pd.to_numeric(df_weather["풍속"], errors="coerce")

df = pd.merge(df_air, df_weather, on="일시", how="inner").dropna(subset=["PM10", "풍속"])
print(f"\n병합 완료: {len(df)}행")
df.sort_values("일시").head()

## 셀 3. 결과 확인: 상관관계 + 시각화 + 예방 메시지

In [ ]:
# -------------------- 3-1. 상관관계 확인 --------------------
corr_wind = df["PM10"].corr(df["풍속"])
corr_rain = df["PM10"].corr(df["강수량"])
print(f"PM10-풍속 상관계수: {corr_wind:.2f}  (음수일수록 '바람이 강할수록 미세먼지가 낮다'는 경향)")
print(f"PM10-강수량 상관계수: {corr_rain:.2f}  (음수일수록 '비가 많이 올수록 미세먼지가 낮다'는 경향)")

# -------------------- 3-2. 산점도: 풍속 vs PM10 --------------------
plt.figure(figsize=(7, 5))
plt.scatter(df["풍속"], df["PM10"], alpha=0.5, color="#2980b9")
if df["풍속"].nunique() > 1:
    z = np.polyfit(df["풍속"], df["PM10"], 1)
    x_line = np.linspace(df["풍속"].min(), df["풍속"].max(), 50)
    plt.plot(x_line, np.poly1d(z)(x_line), color="#c0392b", linewidth=2, label="추세선")
    plt.legend()
plt.xlabel("풍속 (m/s)")
plt.ylabel("PM10 (㎍/m³)")
plt.title("풍속과 미세먼지(PM10)의 관계")
plt.grid(alpha=0.3)
plt.savefig("풍속_PM10_산점도.png", dpi=150, bbox_inches="tight")
plt.show()

# -------------------- 3-3. 비 온 날 vs 비 안 온 날 평균 PM10 --------------------
df["비여부"] = df["강수량"].apply(lambda x: "비 온 시간" if x > 0 else "비 안 온 시간")
summary = df.groupby("비여부")["PM10"].mean()
print("\n=== 강수 유무별 평균 PM10 ===")
print(summary)

plt.figure(figsize=(5, 5))
summary.plot(kind="bar", color=["#7f8c8d", "#2980b9"])
plt.ylabel("평균 PM10 (㎍/m³)")
plt.title("비 온 시간 vs 비 안 온 시간 평균 미세먼지")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.savefig("강수유무_PM10_비교.png", dpi=150, bbox_inches="tight")
plt.show()

# -------------------- 3-4. 예방·대비 메시지 자동 생성 (최근 5개 시각) --------------------
def pm10_message(pm):
    if pm >= 150:
        return "매우 나쁨 — 외출 자제, 마스크(KF80 이상) 착용 권장"
    elif pm >= 80:
        return "나쁨 — 장시간 실외활동 자제, 환기는 짧게"
    elif pm >= 30:
        return "보통 — 민감군(어린이·노약자·호흡기질환자)은 주의"
    else:
        return "좋음 — 환기하기 좋은 시간"

print("\n=== 최근 시각 예방·대비 메시지 ===")
for _, row in df.sort_values("일시").tail(5).iterrows():
    print(f"{row['일시']} | PM10 {row['PM10']:.0f} | 풍속 {row['풍속']:.1f}m/s | {pm10_message(row['PM10'])}")

# -------------------- 3-5. CSV로 결과 저장 --------------------
df.to_csv("미세먼지_기상_병합결과.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료: 미세먼지_기상_병합결과.csv  (왼쪽 폴더 아이콘 → 파일 목록에서 우클릭 → 다운로드)")

## 부록 (선택): 자주 만나는 문제

| 문제 상황 | 원인 | 해결 |
| --- | --- | --- |
| `KeyError: '측정일시'` 같은 컬럼 오류 | 실제 파일의 컬럼명이 다름 | 셀 2의 `head()` 출력에서 실제 컬럼명 확인 → `AIR_COL_MAP`/`WEATHER_COL_MAP`의 왼쪽 값만 수정 |
| 병합 후 `len(df)`가 0 | 두 파일의 시간 형식이 다르거나(예: `2026-01-01 09:00` vs `202601010900`), 기간이 겹치지 않음 | `pd.to_datetime(..., format="...")`으로 형식을 맞추거나, 두 파일의 기간·지역을 다시 확인 |
| 그래프 글자가 네모(ㅁ)로 깨짐 | 폰트 캐시 문제 | 메뉴 "런타임 → 런타임 다시 시작" 후 전체 재실행 |
| 상관계수가 예상과 반대로 나옴 | 실제로 그럴 수 있음(예: 정체된 대기 vs 계절적 요인 등 다른 변수 개입) | 나쁜 결과가 아니라 좋은 발견 — "왜 그런지"를 다음 분석 질문으로 이어가면 됨 |
